
- find the same categories
- merge them into one defintino 
- get final graph


In [3]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import pickle

from collections import defaultdict

In [3]:
cwd = os.getcwd()

parent_dir = Path(cwd).parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

print(sys.path)

['/Users/gl665/.pyenv/versions/3.11.8/lib/python311.zip', '/Users/gl665/.pyenv/versions/3.11.8/lib/python3.11', '/Users/gl665/.pyenv/versions/3.11.8/lib/python3.11/lib-dynload', '', '/Users/gl665/Documents/Phrame-MentalMaps-1/.venv/lib/python3.11/site-packages', '/Users/gl665/Documents/Phrame-MentalMaps-1/notebook']


In [4]:
df = pd.read_csv(
    "/Users/gl665/Documents/Phrame-MentalMaps-1/data/predicted_results/all_variables_pairs_with_category.csv")

print(df.size)
print(df.head())

9197760
   Unnamed: 0                               sentence1  \
0           0  using alcohol - in reference to misuse   
1           1  using alcohol - in reference to misuse   
2           2  using alcohol - in reference to misuse   
3           3  using alcohol - in reference to misuse   
4           4  using alcohol - in reference to misuse   

                                           sentence2  category  
0  perpetration or victimization of violence by a...       NaN  
1  "Increases" in mental health indicate a declin...       NaN  
2  the occurrence of a (physical mostly) violent ...       NaN  
3  perpetration and or victimization of violence ...       NaN  
4               reporting an IPV event to the police       NaN  


# Graph with Disjoint Set (Union-Find) to group the same categories into one group

In [5]:
# grouping if category is the same

class UnionFind:
    def __init__(self):
        self.parent = {}
        self.rank = {}

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression
        return self.parent[x]

    def union(self, x, y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.parent[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.parent[root_x] = root_y
            else:
                self.parent[root_y] = root_x
                self.rank[root_x] += 1

    def add(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0


def group_sentences(pairs):
    uf = UnionFind()

    for sent1, sent2, label in pairs:
        if label == 1:
            uf.add(sent1)
            uf.add(sent2)
            uf.union(sent1, sent2)

    groups = defaultdict(list)
    for sentence in uf.parent.keys():
        root = uf.find(sentence)
        groups[root].append(sentence)

    return list(groups.values())

In [11]:
sentence_pairs = df[['sentence1', 'sentence2', 'category']].to_numpy()

sentence_pairs

array([['using alcohol - in reference to misuse',
        'perpetration or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means',
        nan],
       ['using alcohol - in reference to misuse',
        '"Increases" in mental health indicate a decline in negative emotions and mental illness and a rise in positive emotionality',
        nan],
       ['using alcohol - in reference to misuse',
        'the occurrence of a (physical mostly) violent incident that may warrant police involvement',
        nan],
       ...,
       ['parent reunifying with child after a child welfare case where they have their parental rights back',
        'perpetration or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means',
        nan],
       ['parent reunifying with child after a child welfare case where they have their parental rights back',
        'star

In [12]:
clusters = group_sentences(sentence_pairs)

for i, group in enumerate(clusters):
    print(f"Group {i+1}: {group}")

Group 1: ['resources provided by justice systems to keep victims of violence safe from harm and retaliation of violent romantic partners or enablers']
Group 2: ['resources within justice system to prosecute the perpetratorr, for example, police force able to work on case, capacity for lawyers to work on cases']
Group 3: ['the need of intimate partner violence victims for practical and emotional resources and support ', 'the need for instrumental and or emotional support from others']
Group 4: ['intimate partner violence victims receiving instrumental and emotional resources provided by social or public services']
Group 5: ['the extent to which resources exist to help and support victims of violence heal, recover, and escape violent situations']
Group 6: ['the active pursuit or provision of and obtaining victim resources']
Group 7: ['the extent to which resources exist to help and support healing from trauma', 'the availability of mental health treatment spots at any given time', 'the e

In [4]:
# read in the clds dictionary that includes both the vars name and the graph structure


with open("/Users/gl665/Documents/Phrame-MentalMaps-1/data/data_dictionary/clds_dictionary.pkl", "rb") as f:
    cld_dict = pickle.load(f)

In [8]:
cld_dict[1]['cld_graph'].nodes()

my_dict = cld_dict[1]['node_var_def_dict']

In [19]:
my_dict['alcohol misuse'][0][1]

'misusing alcohol'

In [58]:
def print_graph(G):

    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G)  # Store layout to keep it consistent
    nx.draw_networkx_nodes(G, pos=pos, node_size=20)
    nx.draw_networkx_labels(G, pos=pos, font_size=6)
    nx.draw_networkx_edges(G, pos=pos)  # Added edge drawing

    plt.show()  # Return the figure object

In [63]:
for i in cld_dict:
    nxgraph = cld_dict[i]['cld_graph']
    var_name_to_def_dict = cld_dict[i]['node_var_def_dict']
    node_def = {node: var_name_to_def_dict[node][0][1]
                for node in var_name_to_def_dict.keys()}

    # print(node_def)

    # print(nxgraph.nodes())

    # print_graph(nxgraph)
    def_nxgraph = nx.relabel_nodes(nxgraph, node_def, copy=True)

    # save definition graph
    cld_dict[i]['def_graph'] = def_nxgraph

    # print(def_nxgraph.nodes())

    # print_graph(def_nxgraph)
    print("---------------------------------------")

---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------


In [ ]:
# swap the nodes in the graph if it is in the cluster



In [ ]:
# merge the CLDS!!!!!